# Diagnóstico Bronze — twitch_top_200_games_data

Tabla analizada: `steam_game_intelligence.bronze.twitch_top_200_games_data`. Se comprueban granularidad mensual, rangos de métricas, rankings duplicados y valores ausentes. Se respetan los nombres de columnas del esquema, incluidos los nombres con mayúsculas.

> Ejecuta las celdas en orden. Estas consultas son de diagnóstico: no modifican las tablas Bronze.

## 1. Volumen y cobertura

In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT Game) AS distinct_games,
  COUNT(DISTINCT Year) AS distinct_years,
  MIN(Year) AS min_year,
  MAX(Year) AS max_year,
  MIN(Month) AS min_month,
  MAX(Month) AS max_month,
  MIN(_ingested_at) AS first_ingested_at,
  MAX(_ingested_at) AS last_ingested_at
FROM steam_game_intelligence.bronze.twitch_top_200_games_data;

## 2. Nulos por columna

In [ ]:
%sql
SELECT column_name, null_count
FROM (
  SELECT
    COUNT_IF(`Rank` IS NULL) AS Rank,
    COUNT_IF(Game IS NULL) AS Game,
    COUNT_IF(Month IS NULL) AS Month,
    COUNT_IF(Year IS NULL) AS Year,
    COUNT_IF(Hours_watched IS NULL) AS Hours_watched,
    COUNT_IF(Hours_streamed IS NULL) AS Hours_streamed,
    COUNT_IF(Peak_viewers IS NULL) AS Peak_viewers,
    COUNT_IF(Peak_channels IS NULL) AS Peak_channels,
    COUNT_IF(Streamers IS NULL) AS Streamers,
    COUNT_IF(Avg_viewers IS NULL) AS Avg_viewers,
    COUNT_IF(Avg_channels IS NULL) AS Avg_channels,
    COUNT_IF(Avg_viewer_ratio IS NULL) AS Avg_viewer_ratio,
    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
    COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.twitch_top_200_games_data
) UNPIVOT (null_count FOR column_name IN (
  `Rank`, Game, Month, Year, Hours_watched, Hours_streamed, Peak_viewers,
  Peak_channels, Streamers, Avg_viewers, Avg_channels, Avg_viewer_ratio,
  _ingested_at, _source
))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Validación de año y mes

In [ ]:
%sql
SELECT
  COUNT_IF(Year IS NOT NULL AND (Year < 2016 OR Year > 2023)) AS year_outside_expected_range,
  COUNT_IF(Month IS NOT NULL AND (Month < 1 OR Month > 12)) AS invalid_month,
  COUNT_IF(`Rank` IS NOT NULL AND `Rank` < 1) AS invalid_rank,
  COUNT_IF(`Rank` IS NOT NULL AND `Rank` > 200) AS rank_above_top200,
  COUNT_IF(Game IS NOT NULL AND TRIM(Game) = '') AS blank_game,
  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp
FROM steam_game_intelligence.bronze.twitch_top_200_games_data;

## 4. Duplicados de ranking dentro del mes

In [ ]:
%sql
SELECT Year, Month, `Rank`, COUNT(*) AS row_count,
       COUNT(DISTINCT Game) AS distinct_games
FROM steam_game_intelligence.bronze.twitch_top_200_games_data
GROUP BY Year, Month, `Rank`
HAVING COUNT(*) > 1
ORDER BY Year, Month, `Rank`
LIMIT 200;

## 5. Juegos repetidos en un mismo mes

In [ ]:
%sql
SELECT Year, Month, LOWER(TRIM(Game)) AS normalized_game,
       COUNT(*) AS row_count,
       COUNT(DISTINCT `Rank`) AS distinct_ranks,
       MIN(`Rank`) AS best_rank,
       MAX(`Rank`) AS worst_rank
FROM steam_game_intelligence.bronze.twitch_top_200_games_data
WHERE Game IS NOT NULL AND TRIM(Game) <> ''
GROUP BY Year, Month, LOWER(TRIM(Game))
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 200;

## 6. Métricas negativas o inconsistentes

In [ ]:
%sql
SELECT
  COUNT_IF(Hours_watched < 0) AS negative_hours_watched,
  COUNT_IF(Hours_streamed < 0) AS negative_hours_streamed,
  COUNT_IF(Peak_viewers < 0) AS negative_peak_viewers,
  COUNT_IF(Peak_channels < 0) AS negative_peak_channels,
  COUNT_IF(Streamers < 0) AS negative_streamers,
  COUNT_IF(Avg_viewers < 0) AS negative_avg_viewers,
  COUNT_IF(Avg_channels < 0) AS negative_avg_channels,
  COUNT_IF(Avg_viewer_ratio < 0) AS negative_avg_viewer_ratio
FROM steam_game_intelligence.bronze.twitch_top_200_games_data;

In [ ]:
%sql
SELECT *
FROM steam_game_intelligence.bronze.twitch_top_200_games_data
WHERE `Rank` < 1 OR `Rank` > 200
   OR Month < 1 OR Month > 12
   OR Year < 2016 OR Year > 2023
   OR Hours_watched < 0 OR Hours_streamed < 0
   OR Peak_viewers < 0 OR Peak_channels < 0
   OR Streamers < 0 OR Avg_viewers < 0 OR Avg_channels < 0
   OR Avg_viewer_ratio < 0
   OR Game IS NULL OR TRIM(Game) = ''
LIMIT 200;

## 7. Cobertura por año y mes

In [ ]:
%sql
SELECT Year, Month,
       COUNT(*) AS rows_in_month,
       COUNT(DISTINCT Game) AS distinct_games,
       MIN(`Rank`) AS best_rank,
       MAX(`Rank`) AS worst_rank
FROM steam_game_intelligence.bronze.twitch_top_200_games_data
GROUP BY Year, Month
ORDER BY Year, Month;